# 06 — Tile Training Data Generation

**Goal:** Extract labeled tiles from board photos using the v2 pipeline, then augment to create a balanced training dataset for retraining BoggleCNN.

**Why:** The current CNN was trained on v0-extracted tiles (different framing/perspective), but inference runs on v2-extracted tiles → distribution mismatch → 86.8% accuracy.

**Inputs:**
- `data/labeled-raw/labeled-boards.csv` — 40 labeled boards
- `data/labeled-raw/*.png` — Board images
- Pipeline from `prototyping/pipeline/`

**Outputs:**
- `data/training-data-v2/tiles_raw.npy` — Raw extracted tiles (before augmentation)
- `data/training-data-v2/labels_raw.npy` — Corresponding class indices
- `data/training-data-v2/tiles_augmented.npy` — Balanced + rotated tiles
- `data/training-data-v2/labels_augmented.npy` — Corresponding class indices
- `data/training-data-v2/tile_metadata.csv` — Per-tile provenance

**Usage:** Run all cells in order. No outputs from previous notebooks are required.

## A. Setup

In [ ]:
import sys
import random
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
from ultralytics import YOLO

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch {torch.__version__}, OpenCV {cv2.__version__}")

In [ ]:
PROJECT_ROOT = Path.cwd().parent / "prototyping"
DATA_DIR = PROJECT_ROOT / "data"
LABELED_RAW_DIR = DATA_DIR / "labeled-raw"
LABELED_CSV = LABELED_RAW_DIR / "labeled-boards.csv"
LEGACY_DIR = PROJECT_ROOT / "legacy"
OUTPUT_DIR = DATA_DIR / "training-data-v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_GRID = 6

print(f"Labeled CSV: {LABELED_CSV} (exists: {LABELED_CSV.exists()})")
print(f"Output dir: {OUTPUT_DIR}")

# ── Pipeline (shared module) ──────────────────────────────────────────────────
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline import (
    CLASS_LABELS,
    detect_board, cleanup_mask, fit_quad, warp_board,
    find_tile_centers, infer_grid_from_centroids, extract_tiles_from_grid,
    correct_tile_perspective, preprocess_tile_v0,
    BoggleCNN, predict_tiles_batch,
)
print(f"Pipeline loaded — {len(CLASS_LABELS)} classes: {CLASS_LABELS}")

In [ ]:
yolo_model = YOLO(str(PROJECT_ROOT / "yolov8s-seg.pt"))
print("YOLO loaded.")

## B. Load Labeled Data

In [ ]:
def resolve_labeled_path(raw_path):
    """Normalize Windows-style CSV path to actual file in labeled-raw/."""
    return LABELED_RAW_DIR / Path(raw_path.replace("\\", "/")).name


def parse_letter_sequence(seq):
    """Split semicolon-delimited sequence, stripping empty tokens (trailing ;)."""
    return [s.strip() for s in seq.split(";") if s.strip()]


df = pd.read_csv(LABELED_CSV)
df["img_path"] = df["file_path"].map(resolve_labeled_path)
df["gt_labels"] = df["letter_sequence"].map(parse_letter_sequence)
df["img_exists"] = df["img_path"].map(lambda p: p.exists())
df["n_tiles"] = df["gt_labels"].map(len)

print(f"Loaded {len(df)} labeled boards")
print(f"  Difficulty breakdown: {df['difficulty'].value_counts().to_dict()}")
print(f"  Images found: {df['img_exists'].sum()}/{len(df)}")
print(f"  Tile counts: {df['n_tiles'].value_counts().sort_index().to_dict()}")

# Verify all labels are in CLASS_LABELS
all_labels = {l for labels in df["gt_labels"] for l in labels}
unknown = all_labels - set(CLASS_LABELS)
if unknown:
    print(f"  WARNING: unknown labels not in CLASS_LABELS: {unknown}")
else:
    print(f"  All {len(all_labels)} label classes recognized")

df.head()

## C. Extract Tiles from All Boards

Run the v2 pipeline (detect → warp → grid → extract → perspective correct → preprocess) on each labeled board. Skip boards that fail any stage.

In [ ]:
extraction_records = []
skipped_boards = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting tiles"):
    img_path = row["img_path"]
    gt_labels = row["gt_labels"]

    image = cv2.imread(str(img_path))
    if image is None:
        skipped_boards.append((img_path.name, "imread failed"))
        continue

    # Stage 1–2: detect + clean mask
    mask, box, det_conf = detect_board(image, yolo_model)
    if mask is None:
        skipped_boards.append((img_path.name, "no board detected"))
        continue
    clean = cleanup_mask(mask)

    # Stage 3–4: quad fit + warp
    corners, quad_method = fit_quad(clean)
    if corners is None:
        skipped_boards.append((img_path.name, "quad fitting failed"))
        continue
    warped, warp_sz = warp_board(image, corners)

    # Stage 5: tile center detection + grid inference
    centroids, _ = find_tile_centers(warped)
    if len(centroids) < 4:
        skipped_boards.append((img_path.name, f"only {len(centroids)} peaks"))
        continue
    gs, rows, cols, tsize = infer_grid_from_centroids(centroids, warped.shape)
    if gs == 0:
        skipped_boards.append((img_path.name, "grid inference failed"))
        continue
    if gs != EXPECTED_GRID:
        skipped_boards.append((img_path.name, f"grid {gs}x{gs}, expected {EXPECTED_GRID}"))
        continue

    # Stage 6: extract + perspective correct + preprocess
    raw_tiles = extract_tiles_from_grid(warped, rows, cols, tsize, centroids=centroids)
    tile_imgs = [correct_tile_perspective(t) for t in raw_tiles]
    preprocessed = [preprocess_tile_v0(t) for t in tile_imgs]

    if len(preprocessed) != len(gt_labels):
        skipped_boards.append((
            img_path.name,
            f"tile count mismatch: got {len(preprocessed)}, expected {len(gt_labels)}"
        ))
        continue

    for tile_idx, (tile, label) in enumerate(zip(preprocessed, gt_labels)):
        label_idx = CLASS_LABELS.index(label)
        extraction_records.append({
            "tile": tile,
            "label": label,
            "label_idx": label_idx,
            "board_file": img_path.name,
            "tile_position": tile_idx,
        })

n_ok = len(df) - len(skipped_boards)
print(f"\nExtracted {len(extraction_records)} tiles from {n_ok}/{len(df)} boards")
if skipped_boards:
    print(f"Skipped {len(skipped_boards)} boards:")
    for name, reason in skipped_boards:
        print(f"  {name}: {reason}")

## D. Visual Verification

Spot-check 4 random boards: display extracted tiles in a 6×6 grid with CSV labels overlaid. Verify tile ordering matches ground truth (top-left tile = first CSV label, etc.).

In [ ]:
successful_boards = sorted(set(r["board_file"] for r in extraction_records))
check_boards = random.sample(successful_boards, min(4, len(successful_boards)))

for board_name in check_boards:
    board_tiles = [r for r in extraction_records if r["board_file"] == board_name]
    board_tiles.sort(key=lambda r: r["tile_position"])
    gs = EXPECTED_GRID

    fig, axes = plt.subplots(gs, gs, figsize=(12, 12))
    fig.suptitle(f"Verification: {board_name}", fontsize=14)

    for idx, rec in enumerate(board_tiles):
        r, c = divmod(idx, gs)
        axes[r, c].imshow(rec["tile"], cmap="gray")
        color = "orange" if rec["label"] == "BLOCK" else "green"
        axes[r, c].set_title(rec["label"], fontsize=10, fontweight="bold", color=color)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.show()

## E. Numerical Sanity Check

Run the v0 CNN on the extracted tiles. Accuracy should be ~86.8% (matching notebook 05), confirming the extraction pipeline is identical.

In [ ]:
cnn_v0 = BoggleCNN()
cnn_v0.load_state_dict(
    torch.load(LEGACY_DIR / "models" / "boggle_cnn.pth", map_location="cpu", weights_only=True)
)
cnn_v0.eval()

all_tiles = [r["tile"] for r in extraction_records]
all_gt = [r["label"] for r in extraction_records]

pred_letters, pred_confs = predict_tiles_batch(cnn_v0, all_tiles)
n_correct = sum(p.upper() == g.upper() for p, g in zip(pred_letters, all_gt))
acc = n_correct / len(all_gt)

print(f"v0 CNN accuracy on extracted tiles: {acc:.3f} ({n_correct}/{len(all_gt)})")
print(f"Expected ~0.868 (matching notebook 05 on OK boards)")

del cnn_v0  # free memory

## F. Save Raw Tiles

In [ ]:
tiles_raw = np.stack([r["tile"] for r in extraction_records])   # (N, 100, 100) uint8
labels_raw = np.array([r["label_idx"] for r in extraction_records])  # (N,) int

np.save(OUTPUT_DIR / "tiles_raw.npy", tiles_raw)
np.save(OUTPUT_DIR / "labels_raw.npy", labels_raw)

print(f"Saved raw tiles: {tiles_raw.shape} ({tiles_raw.nbytes / 1024 / 1024:.1f} MB)")
print(f"Saved raw labels: {labels_raw.shape}")
print(f"Output: {OUTPUT_DIR}")

## G. Class Distribution (Before Augmentation)

In [ ]:
raw_counts = Counter(r["label"] for r in extraction_records)

# Sort by CLASS_LABELS order
labels_ordered = [l for l in CLASS_LABELS if l in raw_counts]
counts_ordered = [raw_counts[l] for l in labels_ordered]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#f44336" if c < 10 else "#ff9800" if c < 30 else "#4caf50" for c in counts_ordered]
bars = ax.bar(labels_ordered, counts_ordered, color=colors, edgecolor="black", linewidth=0.5)
for bar, c in zip(bars, counts_ordered):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(c), ha="center", fontsize=7)
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title(f"Raw Class Distribution ({len(extraction_records)} tiles, {len(raw_counts)} classes)")
plt.tight_layout()
plt.show()

max_cls = max(raw_counts, key=raw_counts.get)
min_cls = min(raw_counts, key=raw_counts.get)
print(f"Max class: {raw_counts[max_cls]} ({max_cls})")
print(f"Min class: {raw_counts[min_cls]} ({min_cls})")
print(f"Imbalance ratio: {raw_counts[max_cls] / raw_counts[min_cls]:.1f}x")

## H. Class Balancing via Duplication

Duplicate underrepresented classes up to `max_class_count × 2`. Matches the v0 training strategy.

In [ ]:
MAX_CLASS_MULT = 2
max_count = max(raw_counts.values())
target_per_class = max_count * MAX_CLASS_MULT

print(f"Max class count: {max_count}")
print(f"Target per class: {target_per_class}")

balanced_records = list(extraction_records)  # start with all originals
dup_count = 0

for label in CLASS_LABELS:
    class_tiles = [r for r in extraction_records if r["label"] == label]
    if not class_tiles:
        print(f"  WARNING: no tiles for class '{label}'")
        continue

    n_needed = target_per_class - len(class_tiles)
    if n_needed <= 0:
        continue

    for i in range(n_needed):
        source = class_tiles[i % len(class_tiles)]
        balanced_records.append({
            **source,
            "augmentation": f"dup_{i}",
        })
        dup_count += 1

print(f"\nAfter balancing: {len(balanced_records)} tiles "
      f"(was {len(extraction_records)}, added {dup_count} duplicates)")
print(f"Target per class: {target_per_class} → total: {target_per_class * len(CLASS_LABELS)}")

## I. Rotation Augmentation

Apply 0°/90°/180°/270° to every tile. Uses `np.rot90` for lossless rotation of the binary masks.

In [ ]:
ROTATIONS = [0, 90, 180, 270]

augmented_tiles = []
augmented_labels = []
metadata_records = []

for rec in tqdm(balanced_records, desc="Rotation augmentation"):
    tile = rec["tile"]
    label_idx = rec["label_idx"]

    for angle in ROTATIONS:
        if angle == 0:
            rotated = tile
        else:
            rotated = np.rot90(tile, k=angle // 90)

        augmented_tiles.append(rotated)
        augmented_labels.append(label_idx)
        metadata_records.append({
            "label": rec["label"],
            "label_idx": label_idx,
            "board_file": rec["board_file"],
            "tile_position": rec["tile_position"],
            "rotation": angle,
            "augmentation": rec.get("augmentation", "original"),
        })

tiles_aug = np.stack(augmented_tiles)    # (N_aug, 100, 100) uint8
labels_aug = np.array(augmented_labels)  # (N_aug,) int

print(f"Final augmented dataset: {tiles_aug.shape}")
print(f"Memory: {tiles_aug.nbytes / 1024 / 1024:.1f} MB")
print(f"Expected tiles per class: {target_per_class * len(ROTATIONS)}")

## J. Class Distribution (After Augmentation)

In [ ]:
aug_counts = Counter(r["label"] for r in metadata_records)
labels_ordered_aug = [l for l in CLASS_LABELS if l in aug_counts]
counts_ordered_aug = [aug_counts[l] for l in labels_ordered_aug]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(labels_ordered_aug, counts_ordered_aug, color="#4caf50", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title(f"Augmented Class Distribution ({len(metadata_records)} tiles, balanced)")
plt.tight_layout()
plt.show()

unique_counts = set(aug_counts.values())
print(f"Unique counts per class: {unique_counts}")
if len(unique_counts) == 1:
    print("All classes perfectly balanced!")
else:
    print("WARNING: classes are not perfectly balanced")

## K. Save Augmented Dataset

In [ ]:
np.save(OUTPUT_DIR / "tiles_augmented.npy", tiles_aug)
np.save(OUTPUT_DIR / "labels_augmented.npy", labels_aug)

metadata_df = pd.DataFrame(metadata_records)
metadata_df.to_csv(OUTPUT_DIR / "tile_metadata.csv", index=False)

print(f"Saved to {OUTPUT_DIR}:")
print(f"  tiles_augmented.npy:  {tiles_aug.shape} ({tiles_aug.nbytes / 1024 / 1024:.1f} MB)")
print(f"  labels_augmented.npy: {labels_aug.shape}")
print(f"  tile_metadata.csv:    {len(metadata_df)} rows")

## L. Summary

In [ ]:
print("=" * 60)
print("  TILE TRAINING DATA GENERATION SUMMARY")
print("=" * 60)
print(f"  Labeled boards:         {len(df)}")
print(f"  Boards processed (OK):  {len(df) - len(skipped_boards)}")
print(f"  Boards skipped:         {len(skipped_boards)}")
print(f"  Raw tiles extracted:    {len(extraction_records)}")
print(f"  Classes represented:    {len(raw_counts)}/{len(CLASS_LABELS)}")
print(f"  Target per class:       {target_per_class}")
print(f"  Tiles after balancing:  {len(balanced_records)}")
print(f"  Rotations applied:      {ROTATIONS}")
print(f"  Final dataset size:     {tiles_aug.shape[0]}")
print(f"  Final dataset shape:    {tiles_aug.shape}")
print(f"  Final dataset memory:   {tiles_aug.nbytes / 1024 / 1024:.1f} MB")
print("=" * 60)
if skipped_boards:
    print("\nSkipped boards:")
    for name, reason in skipped_boards:
        print(f"  {name}: {reason}")